<a href="https://colab.research.google.com/github/IshikaGeed/HackOweek/blob/main/HackOWeeek8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

dates = pd.date_range(start="2025-01-01", periods=500, freq="H")

cars = np.random.poisson(lam=20, size=len(dates))

df = pd.DataFrame({
    "timestamp": dates,
    "cars_count": cars
})

df.head()

/tmp/ipykernel_515/1778330679.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start="2025-01-01", periods=500, freq="H")


,timestamp,cars_count
0,2025-01-01 00:00:00,16
1,2025-01-01 01:00:00,23
2,2025-01-01 02:00:00,25
3,2025-01-01 03:00:00,18
4,2025-01-01 04:00:00,23


In [4]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1LzEOr8pyEhGgQymGnaDV5cXRQEp9t4qb2Gomri2yN8c/edit#gid=0


In [5]:
def categorize(x):
    if x < 10:
        return "Low"
    elif x < 25:
        return "Medium"
    else:
        return "High"

df["category"] = df["cars_count"].apply(categorize)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["cat_encoded"] = le.fit_transform(df["category"])

X = df[["cars_count"]]
y = df["cat_encoded"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

model = GaussianNB()
model.fit(X_train,y_train)

GaussianNB()

In [7]:
model.predict([[18]])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(


array([1])

In [8]:
!pip install prophet

In [9]:
from prophet import Prophet

prophet_df = df.rename(columns={
    "timestamp":"ds",
    "cars_count":"y"
})

m = Prophet()
m.fit(prophet_df)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.


In [10]:
future = m.make_future_dataframe(periods=48,freq='H')

forecast = m.predict(future)
forecast[['ds','yhat']].tail()

/usr/local/lib/python3.12/dist-packages/prophet/forecaster.py:1875: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


,ds,yhat
543,2025-01-23 15:00:00,21.630923
544,2025-01-23 16:00:00,21.535301
545,2025-01-23 17:00:00,21.200528
546,2025-01-23 18:00:00,21.123492
547,2025-01-23 19:00:00,21.424910


In [11]:
def lighting(x):
    if x < 10:
        return 30
    elif x < 25:
        return 70
    else:
        return 100

forecast["lighting"] = forecast["yhat"].apply(lighting)

In [12]:
!pip install gradio

In [13]:
import gradio as gr

def forecast_lighting(hour_index):

    row = forecast.iloc[int(hour_index)]

    return {
        "Time": str(row["ds"]),
        "Predicted Cars": int(row["yhat"]),
        "Lighting %": row["lighting"]
    }

slider = gr.Slider(0,len(forecast)-1,step=1)

interface = gr.Interface(
    fn=forecast_lighting,
    inputs=slider,
    outputs="json",
    title="Parking Lot Lighting Forecast"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42fb73b6c7f1203349.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
